# 0917 14일차

## 1. Conv2D의 파라미터 개수

Conv2D가 학습하는 가중치(w)와 bias(b)의 개수

![커널 c장과 bias가 필터 1개로 묶이고 f개로 늘어나는 과정](assets/conv2d-param-count.gif)
![Conv2D 파라미터 개수 계산 과정](assets/conv2d-param-count.svg)

### 1-1. 계산 과정

1. 커널 한 장 : `n × n`
   - 커널이 `(3, 3)`이면 가중치 9개
2. 입력 채널 수만큼 : `n × n × c`
   - 필터 하나가 입력 채널 `c`장을 한꺼번에 보므로 채널마다 커널이 한 장씩 있음
3. bias 더하기 : `n × n × c + b`
   - bias는 채널 수와 상관없이 **필터 하나에 1개** → 계산할 때 `b = 1`
   - 여기까지가 필터 1개의 파라미터
4. 필터 개수만큼 : `f × (n × n × c + b)`

```
파라미터 = f × (n × n × c + b)

f : filters (이 층의 필터 개수)
n : kernel_size
c : 입력 채널 수
b : bias (필터당 1개)
```

- `use_bias=False`면 `+ b`가 빠져 `f × n × n × c`가 됨

### 1-2. 입력 채널 수 c

1. 첫 층 : 이미지의 채널 → 흑백 1, 컬러 3
2. 둘째 층부터 : **앞 층의 filters**
   - 앞 층이 필터 64개로 특징맵 64장을 만들었으니, 다음 층의 필터는 64장을 한꺼번에 봄

| 층 | 코드 | c | 파라미터 |
|---|---|---|---|
| 첫 층 | `Conv2D(64, (3, 3), input_shape=(32, 32, 3))` | 3 | `64 × (3×3×3 + 1)` = 1,792 |
| 둘째 층 | `Conv2D(64, (3, 3))` | 64 | `64 × (3×3×64 + 1)` = 36,928 |

- 같은 `(3, 3)` 커널인데 둘째 층이 20배 많은 이유는 `c`가 3에서 64로 커졌기 때문임

**케라스가 실제로 저장하는 모양**

```python
w, b = model.layers[0].get_weights()
print(w.shape)   # (3, 3, 3, 64)   <- (n, n, c, f)
print(b.shape)   # (64,)           <- (f,)
```

- 커널 모양 `(n, n, c, f)`를 모두 곱하고 bias `f`개를 더한 것이 파라미터 개수임

### 1-3. 파라미터 개수에 영향을 주지 않는 것

1. 입력 이미지의 가로·세로
   - 필터 하나를 이미지 전체에 옮겨가며 **같은 가중치로** 계산함
   - 그래서 28×28이든 32×32든 필터의 가중치 개수는 같음
   - `Dense`는 입력 값 하나마다 가중치가 따로 있어서 입력이 커지면 가중치도 늘어남
2. `padding`, `strides` : 출력 크기만 바꾸고 필터 크기는 그대로임 (13일차 §3, §4)
3. `MaxPooling2D` : 계산하는 가중치가 없어서 0개 (13일차 §5)

## 2. Global Average Pooling

특징맵 한 장마다 전체 값의 평균 하나만 남기는 층

```
(세로, 가로, 채널)  →  (채널,)
(10, 10, 32)       →  (32,)
```

![Flatten과 GlobalAveragePooling2D에서 Dense로 들어가는 값과 파라미터 비교](assets/global-average-pooling.gif)
![Flatten과 GlobalAveragePooling2D 비교](assets/global-average-pooling.svg)

### 2-1. 필요한 이유 - Flatten의 파라미터 문제

1. `Flatten`은 모든 값을 한 줄로 펴서 `Dense`에 넘김 (13일차 §1)
2. `Dense`의 파라미터는 입력 개수에 비례함 → `(입력 개수 + 1) × 노드 수` (9일차 §2-1)
3. 그래서 Conv 출력이 크면 `Flatten` 바로 뒤 `Dense`의 파라미터가 크게 늘어남

`(10, 10, 32)` 출력 뒤에 `Dense(10)`을 붙이는 경우

| 연결 방법 | Dense에 들어가는 값 | Dense(10) 파라미터 |
|---|---|---|
| `Flatten` | 10 × 10 × 32 = 3,200개 | (3,200 + 1) × 10 = **32,010** |
| `GlobalAveragePooling2D` | 32개 | (32 + 1) × 10 = **330** |

- `Dense(10)`의 파라미터가 32,010개에서 330개로 약 1/100이 됨

### 2-2. 동작

1. 특징맵 한 장(`10 × 10`)의 값을 모두 평균 내서 값 1개로 만듦
2. 이것을 채널마다 반복함 → 출력 개수 = 채널 수 = 앞 Conv 층의 `filters`
3. 결과가 이미 1차원 `(채널,)`이라 `Flatten` 없이 바로 `Dense`에 연결함
4. 평균만 계산하므로 파라미터는 0개

```python
from tensorflow.keras.layers import GlobalAveragePooling2D

model.add(Conv2D(32, (3, 3), activation='relu'))   # (10, 10, 32)
model.add(GlobalAveragePooling2D())                # (32,)      Flatten 대신
model.add(Dense(10, activation='softmax'))         # 파라미터 330
```

#### cf) AveragePooling2D와의 차이

이름이 비슷하지만 평균을 내는 범위가 다름

| 층 | 평균 내는 범위 | `(10, 10, 32)` 입력 → 출력 |
|---|---|---|
| `AveragePooling2D()` | 2×2 구역마다 | `(5, 5, 32)` |
| `GlobalAveragePooling2D()` | 특징맵 한 장 전체 | `(32,)` |

`1 ~ 16`이 든 4×4 특징맵 한 장에 적용한 결과

```
[[ 1  2  3  4]      AveragePooling2D       →  [[ 3.5  5.5]
 [ 5  6  7  8]                                 [11.5 13.5]]
 [ 9 10 11 12]
 [13 14 15 16]]     GlobalAveragePooling2D →  [8.5]
```

- `AveragePooling2D`는 `MaxPooling2D`(13일차 §5)에서 최댓값 대신 평균을 쓰는 것이라 크기를 절반으로 줄이는 용도
- `GlobalAveragePooling2D`는 특징맵을 값 하나로 줄여 `Flatten`을 대신하는 용도

## 3. DNN으로 이미지 처리

`Dense`만 쌓은 모델(DNN)로 이미지를 처리하는 것. 픽셀 하나를 특성 하나로 봄

`Dense`는 `(행, 열)` 2차원을 받으므로 한 장을 한 행으로 폄 (12일차 §3)

```python
x_train = x_train.reshape(-1, 28 * 28)        # 흑백 : (60000, 28, 28) -> (60000, 784)
x_train = x_train.reshape(-1, 32 * 32 * 3)    # 컬러 : (50000, 32, 32, 3) -> (50000, 3072)
model.add(Dense(1024, input_shape=(28 * 28,)))
```

**주의)**
- 컬러는 채널까지 곱해야 함. `32 * 32`로 하면 한 장이 R·G·B 세 행으로 쪼개져 `(150000, 1024)`가 되고 `fit`에서 `Data cardinality is ambiguous`
- `input_shape=(32 * 32)`는 숫자라 `TypeError`. 값이 하나여도 `(3072,)`처럼 쉼표를 붙여야 튜플

## 4. 2차원 데이터를 CNN으로 처리

§3과 반대로 표 데이터 `(n, 특성 수)`를 reshape해서 `Conv2D`에 넣는 것

**가능한 조건**
1. 원소 개수가 같아야 함 → 특성 8개면 `8 = 8×1×1 = 4×2×1 = 2×2×2`
2. `Conv2D`는 4차원 `(n, 세로, 가로, 채널)`. `(n, 8, 1)` 3차원은 `Conv1D`용
3. 세로·가로가 커널 이상이어야 함. 작으면 `padding='same'`

`(n, 8)`을 reshape해서 `Conv2D(64, 커널)`에 넣은 결과

| reshape | 커널 `(2, 1)` | 커널 `(2, 2)` |
|---|---|---|
| `(n, 8, 1, 1)` | `(7, 1, 64)` | 에러 (가로 1 < 2) |
| `(n, 4, 2, 1)` | `(3, 2, 64)` | `(3, 1, 64)` |
| `(n, 2, 2, 2)` | `(1, 2, 64)` | `(1, 1, 64)` |

- reshape만 맞으면 돌아가지만, 성능이 좋아지는지는 돌려봐야 앎
- CNN은 옆에 붙은 값끼리 관련이 있다고 보는데, 표의 열 순서는 사람이 정한 것이라 그런 보장이 없음

#### 주의) 변환할 때 틀리기 쉬운 곳

`keras42_cnn01` ~ `cnn10`에서 실제로 틀렸던 것

1. reshape 곱 ≠ 특성 개수
   - cancer는 특성 30개인데 `(5, 2, 1)`로 펴서 x가 3배가 됨 → `fit`에서 y와 개수 불일치
   - `-1` 때문에 reshape 자체는 에러가 안 남. 먼저 `x.shape`를 확인하고 `30 = 10 × 3`처럼 맞춤
2. 작은 입력에 크기를 줄이는 층
   - california `(4, 2, 1)`은 padding 없는 `Conv2D (2, 2)` 뒤 `(3, 1)`이 되어 `MaxPool2D`에서 에러
   - `padding='same'`을 주고, `MaxPool2D()`는 괄호를 붙여 층으로 넣음 (8일차 §0-1)
3. 출력층 활성화
   - 입력을 CNN으로 바꿔도 출력층 규칙은 그대로 : 회귀 `Dense(1)`, 이진 `Dense(1, 'sigmoid')`, 다중 `Dense(라벨 수, 'softmax')`
   - cancer·santander·wine은 은닉층에 sigmoid·softmax가 붙고 출력층에는 활성화가 없었음 (7일차 §4, 8일차 §4)